# Sports Mathematics × Agent Orchestration
## Interactive Formula Exploration Notebook
> Demos all 6 formulas, 4 normalization methods, ILP optimizer, QUIBIDT / STRATEGA / MANNA layers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(12,5),'axes.facecolor':'#1e293b','figure.facecolor':'#0f172a','text.color':'#f1f5f9','axes.labelcolor':'#f1f5f9','xtick.color':'#94a3b8','ytick.color':'#94a3b8','axes.edgecolor':'#334155','grid.color':'#334155'})
print('Setup complete - dark theme active')

## Formula 1 - Efficiency Score
`Eff_a = Value_a / Cost_a`
Measures agent throughput per unit resource consumed.

In [ ]:
from src.formulas import efficiency_score
agents=[{'name':'Alice','value':45,'cost':8000},{'name':'Bob','value':38,'cost':7200},{'name':'Carol','value':52,'cost':9000},{'name':'Dave','value':29,'cost':6500},{'name':'Eve','value':41,'cost':7800},{'name':'Frank','value':35,'cost':6800}]
eff=[efficiency_score(a['value'],a['cost'])*1000 for a in agents]
names=[a['name'] for a in agents]
colors=['#38bdf8' if e==max(eff) else '#0ea5e9' for e in eff]
plt.bar(names,eff,color=colors,edgecolor='#0369a1')
plt.title('Formula 1: Efficiency Score (Value/Cost x1000)',color='#f1f5f9')
plt.ylabel('Efficiency')
plt.tight_layout(); plt.show()
print(f'Top agent: {names[eff.index(max(eff))]} ({max(eff):.4f})')

## Formula 2 - Expected Value
`EV_a = Σ P(a,j) · V(a,j)`
Probabilistic contribution weighted by scenario outcomes.

In [ ]:
from src.formulas import expected_value
scenarios=[{'name':'Alice','probs':[0.7,0.2,0.1],'outcomes':[100,50,0]},{'name':'Bob','probs':[0.5,0.3,0.2],'outcomes':[90,40,0]},{'name':'Carol','probs':[0.8,0.1,0.1],'outcomes':[110,60,0]},{'name':'Dave','probs':[0.4,0.4,0.2],'outcomes':[80,35,0]},{'name':'Eve','probs':[0.6,0.3,0.1],'outcomes':[95,45,0]},{'name':'Frank','probs':[0.55,0.25,0.2],'outcomes':[85,38,0]}]
ev=[expected_value(s['probs'],s['outcomes']) for s in scenarios]
plt.bar([s['name'] for s in scenarios],ev,color='#a78bfa',edgecolor='#7c3aed')
plt.title('Formula 2: Expected Value per Agent',color='#f1f5f9')
plt.ylabel('EV Score')
plt.tight_layout(); plt.show()

## Formula 3 - ILP Optimizer (Lineup)
`max Σ v_a·x_a  s.t.  Σ c_a·x_a ≤ B,  x_a ∈ {0,1}`
Combinatorial selection of agents within budget.

In [ ]:
from src.optimizer import greedy_knapsack, dp_knapsack
budget=22000
g_sel,g_val,g_cost=greedy_knapsack(agents,budget)
d_sel,d_val,d_cost=dp_knapsack(agents,budget)
print(f'Greedy : {[a["name"] for a in g_sel]} | Value={g_val} | Cost=${g_cost:,}')
print(f'DP/Exact: {[a["name"] for a in d_sel]} | Value={d_val} | Cost=${d_cost:,}')
selected={a['name'] for a in d_sel}
colors=['#4ade80' if n in selected else '#475569' for n in names]
plt.bar(names,[a['value'] for a in agents],color=colors,edgecolor='#1e293b')
plt.title('Formula 3: ILP Selection (green=selected, budget=$22,000)',color='#f1f5f9')
plt.ylabel('Value'); plt.tight_layout(); plt.show()

## Formula 4 - Dynamic Weight
`w_a = α·Eff + β·EV − γ·Risk − δ·Load`
Composite routing score balancing performance against risk and load.

In [ ]:
from src.formulas import compute_dynamic_weight
from src.normalization import softmax
agent_data=[{'name':'Alice','efficiency':0.90,'ev':88,'risk':0.10,'load':0.50},{'name':'Bob','efficiency':0.70,'ev':72,'risk':0.20,'load':0.70},{'name':'Carol','efficiency':0.85,'ev':96,'risk':0.12,'load':0.55},{'name':'Dave','efficiency':0.60,'ev':58,'risk':0.30,'load':0.80},{'name':'Eve','efficiency':0.80,'ev':82,'risk':0.15,'load':0.60},{'name':'Frank','efficiency':0.65,'ev':65,'risk':0.25,'load':0.75}]
raw_w=[compute_dynamic_weight(a,alpha=1.0,beta=0.5,gamma=0.3,delta=0.2) for a in agent_data]
probs=softmax(raw_w)
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].bar([a['name'] for a in agent_data],raw_w,color='#22d3ee',edgecolor='#0e7490')
axes[0].set_title('Raw Dynamic Weights',color='#f1f5f9'); axes[0].set_ylabel('w_a')
axes[1].pie(probs,labels=[a['name'] for a in agent_data],autopct='%1.1f%%',colors=['#38bdf8','#818cf8','#a78bfa','#f472b6','#4ade80','#fb923c'])
axes[1].set_title('Softmax Routing Probabilities',color='#f1f5f9')
plt.tight_layout(); plt.show()

## Formula 5 - Normalization Comparator
All 4 methods side-by-side: Softmax / Temperature Softmax / Min-Max / Z-Score

In [ ]:
from src.normalization import softmax,temperature_softmax,min_max_normalize,z_score_normalize
W=raw_w
methods={'Softmax':softmax(W),'Temp(T=0.3)':temperature_softmax(W,0.3),'Min-Max':min_max_normalize(W),'Z-Score':z_score_normalize(W)}
palette=['#38bdf8','#a78bfa','#4ade80','#fb923c']
fig,axes=plt.subplots(1,4,figsize=(18,5))
for ax,(m,vals),col in zip(axes,methods.items(),palette):
    ax.bar([a['name'] for a in agent_data],vals,color=col,edgecolor='#1e293b')
    ax.set_title(m,color='#f1f5f9'); ax.tick_params(axis='x',rotation=45)
plt.suptitle('Formula 5: Normalization Method Comparison',color='#f1f5f9',y=1.02)
plt.tight_layout(); plt.show()

## Formula 6 - Weight Update Rule (Load Balancing)
`w_a(t+1) = w_a(t) + η·(w_a/Σw)·(L_avg − L_a)`
Adaptive load-balancing converging toward uniform load distribution.

In [ ]:
from src.formulas import weight_update
np.random.seed(42)
T=15
history={a['name']:[compute_dynamic_weight(a,alpha=1.0,beta=0.5,gamma=0.3,delta=0.2)] for a in agent_data}
loads={a['name']:[a['load']] for a in agent_data}
for t in range(T):
    cur_w=[history[a['name']][-1] for a in agent_data]
    cur_l=[loads[a['name']][-1] for a in agent_data]
    l_avg=np.mean(cur_l)
    for i,a in enumerate(agent_data):
        history[a['name']].append(weight_update(cur_w[i],cur_w,l_avg,cur_l[i],eta=0.1))
        loads[a['name']].append(float(np.clip(cur_l[i]+np.random.uniform(-0.05,0.08),0.1,1.0)))
colors_ev=['#38bdf8','#818cf8','#a78bfa','#f472b6','#4ade80','#fb923c']
for (name,vals),col in zip(history.items(),colors_ev):
    plt.plot(range(T+1),vals,label=name,color=col,marker='o',markersize=3)
plt.title('Formula 6: Weight Evolution over 15 Time Steps',color='#f1f5f9')
plt.xlabel('Time Step'); plt.ylabel('w_a(t)'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## End-to-End: AgentOrchestrator Demo
QUIBIDT → STRATEGA → MANNA running together across all routing modes.

In [ ]:
from src.orchestrator import AgentOrchestrator
for mode in ['probabilistic','top_k','threshold']:
    orc=AgentOrchestrator(agent_data,budget=500_000,routing_mode=mode)
    counts={a['name']:0 for a in agent_data}
    for i in range(30):
        r=orc.assign_task(task_value=1000)
        counts[r['name']]+=1
    print(f'Mode: {mode:14s} | {counts}')
print('\nMANNA summary:', orc.manna.summary())

---
*Sports Mathematics × Agent Orchestration | QUIBIDT | STRATEGA | MANNA | Deep Code Integration*